In [ ]:
"""
K-FOLD EVALUATION (With Optimized Parameters)

PURPOSE: Using the best hyperparameters found in Stage 1 (HPO),
run K=5 Cross Validation test for all models (4 Transformers + XGBoost)
and generate final article tables (PHASE 1 and PHASE 2).
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb
from scipy import stats
import time
import json
from pathlib import Path
import warnings
import gc
import copy
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore')

K_FOLDS = 5
CV_RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {DEVICE}")

def set_all_seeds(seed):
    import random
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def reset_gpu_stats():
    if DEVICE.type == 'cuda':
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(DEVICE)

def get_peak_gpu_memory_mb():
    if DEVICE.type == 'cuda':
        return torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    return 0.0

class ImprovedDataProcessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.sales_scaler = StandardScaler()
        self.y_log_max = None
        self.view_scaler = StandardScaler()

    def load_full_dataset(self, excel_path='restaurant_merged_data.xlsx'):
        return pd.read_excel(excel_path, sheet_name='Merged_Data')

    def get_stratify_groups(self, data, test_size_for_rare_calc=0.1):
        category_counts = data['sales_main_category'].value_counts()
        min_samples_needed = int(np.ceil(1 / test_size_for_rare_calc)) + 1
        rare_categories = category_counts[category_counts < min_samples_needed].index.tolist()
        stratify_groups = data['sales_main_category'].copy()
        if len(rare_categories) > 0:
            stratify_groups = stratify_groups.replace(rare_categories, 'RARE_COMBINED')
        return stratify_groups

    def prepare_features_fair(self, data_df, label_encoder, is_train=True):
        categories = data_df['sales_main_category'].fillna('Unknown').astype(str).str.strip()
        category_encoded = np.array([
            label_encoder.transform([cat])[0] if cat in label_encoder.classes_ else
            (label_encoder.transform(['Unknown'])[0] if 'Unknown' in label_encoder.classes_ else 0)
            for cat in categories
        ])
        target = data_df['net_sales_qty'].fillna(0).values
        sales_features = []
        net_sales_amount = data_df['net_sales_amount'].fillna(0).values
        sales_features.append(net_sales_amount); sales_features.append(np.log1p(net_sales_amount))
        if 'gross_sales_amount' in data_df.columns: sales_features.append(data_df['gross_sales_amount'].fillna(0))
        if 'cost' in data_df.columns: sales_features.append(data_df['cost'].fillna(0))
        if 'discount_amount' in data_df.columns: sales_features.append(data_df['discount_amount'].fillna(0))
        view_features = []
        view_count = data_df['view_count'].fillna(0).values
        view_features.append(view_count); view_features.append(np.log1p(view_count))
        if 'view_duration' in data_df.columns: view_features.append(data_df['view_duration'].fillna(0))
        if 'avg_view_duration' in data_df.columns: view_features.append(data_df['avg_view_duration'].fillna(0))
        view_features.append((view_count > 0).astype(int))
        X_sales = np.column_stack(sales_features); X_view = np.column_stack(view_features)
        X_all = np.column_stack([X_sales, X_view]); view_mask = (view_count > 0)
        y = np.maximum(target, 0.1); y_log = np.log1p(y)
        if is_train:
            self.y_log_max = y_log.max()
            if self.y_log_max == 0 or self.y_log_max is None: self.y_log_max = 1.0
        elif self.y_log_max is None: self.y_log_max = 1.0
        y_norm = y_log / self.y_log_max
        return X_sales, X_view, X_all, y_norm, y_log, category_encoded, view_mask

class BaselineTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=2, n_categories=18, dropout=0.1):
        super().__init__()
        self.category_embedding = nn.Embedding(n_categories, 8)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.category_projection = nn.Linear(8, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 1), nn.ReLU()
        )
    def forward(self, x, categories, view_mask=None):
        x_proj = self.input_projection(x); cat_emb = self.category_embedding(categories)
        cat_proj = self.category_projection(cat_emb)
        sequence = torch.stack([x_proj, cat_proj], dim=1)
        out = self.transformer(sequence)
        return self.output(out[:, 0, :]).squeeze(-1)

class DualStreamTransformer(nn.Module):
    def __init__(self, sales_dim, view_dim, d_model=32, nhead=2, n_categories=18, dropout=0.1):
        super().__init__()
        self.category_embedding = nn.Embedding(n_categories, 8)
        self.sales_projection = nn.Linear(sales_dim + 8, d_model)
        self.view_projection = nn.Linear(view_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 1), nn.ReLU()
        )
    def forward(self, x_sales, x_view, categories, view_mask):
        cat_emb = self.category_embedding(categories)
        sales_with_cat = torch.cat([x_sales, cat_emb], dim=1)
        sales_proj = self.sales_projection(sales_with_cat)
        view_proj = self.view_projection(x_view)
        view_mask_expanded = view_mask.unsqueeze(1).float()
        combined = sales_proj + 0.5 * view_proj * view_mask_expanded
        sequence = combined.unsqueeze(1)
        out = self.transformer(sequence)
        return self.output(out.squeeze(1)).squeeze(-1)

class AdaptiveWeightingTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=2, n_categories=18, dropout=0.1, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.category_embedding = nn.Embedding(n_categories, 8)
        self.feature_projection = nn.Linear(input_dim, d_model)
        self.category_projection = nn.Linear(8, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 1), nn.ReLU()
        )
    def forward(self, x, categories, view_mask):
        feat_proj = self.feature_projection(x)
        mask_expanded = view_mask.unsqueeze(1).float()
        weighted = feat_proj * (mask_expanded + (1 - mask_expanded) * self.alpha)
        cat_emb = self.category_embedding(categories)
        cat_proj = self.category_projection(cat_emb)
        sequence = torch.stack([weighted, cat_proj], dim=1)
        out = self.transformer(sequence)
        return self.output(out[:, 0, :]).squeeze(-1)

class ConfigurableEnsembleTransformer(nn.Module):
    def __init__(self, sales_dim, full_dim, d_model=32, nhead=2, num_layers=1,
                 dropout=0.1, n_categories=18, category_emb_dim=8):
        super().__init__()
        self.category_embedding = nn.Embedding(n_categories, category_emb_dim)
        self.sales_projection = nn.Linear(sales_dim + category_emb_dim, d_model)
        sales_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, dropout=dropout, batch_first=True)
        self.sales_transformer = nn.TransformerEncoder(sales_encoder_layer, num_layers)
        self.sales_output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())
        self.full_projection = nn.Linear(full_dim + category_emb_dim, d_model)
        full_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, dropout=dropout, batch_first=True)
        self.full_transformer = nn.TransformerEncoder(full_encoder_layer, num_layers)
        self.full_output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())
        self.gate = nn.Sequential(
            nn.Linear(category_emb_dim + 1, 16), nn.ReLU(), nn.Linear(16, 2), nn.Softmax(dim=1))
    def forward(self, x_sales, x_all, categories, view_mask):
        cat_emb = self.category_embedding(categories)
        sales_input = torch.cat([x_sales, cat_emb], dim=1)
        sales_proj = self.sales_projection(sales_input).unsqueeze(1)
        sales_out = self.sales_transformer(sales_proj)
        sales_pred = self.sales_output(sales_out.squeeze(1))
        full_input = torch.cat([x_all, cat_emb], dim=1)
        full_proj = self.full_projection(full_input).unsqueeze(1)
        full_out = self.full_transformer(full_proj)
        full_pred = self.full_output(full_out.squeeze(1))
        view_flag = view_mask.float().unsqueeze(1)
        gate_input = torch.cat([cat_emb, view_flag], dim=1)
        gate_weights = self.gate(gate_input)
        final_pred = gate_weights[:, 0:1] * sales_pred + gate_weights[:, 1:2] * full_pred
        return final_pred.squeeze(-1)

def get_metrics(y_pred_norm, y_true_log, processor):
    pred_orig = np.expm1(y_pred_norm * processor.y_log_max)
    true_orig = np.expm1(y_true_log)
    pred_orig = np.maximum(pred_orig, 0.1)
    true_orig = np.maximum(true_orig, 0.1)
    r2 = r2_score(true_orig, pred_orig)
    mae = mean_absolute_error(true_orig, pred_orig)
    mape = np.median(np.abs((true_orig - pred_orig) / np.maximum(true_orig, 1))) * 100
    return {'r2': r2, 'mae': mae, 'mape': mape}

def train_script1_model_run(model, processor, train_data, test_data, method_name, seed,
                           label_encoder,
                           lr=0.002, weight_decay=1e-5,
                           epochs=100, verbose=False):
    set_all_seeds(seed); reset_gpu_stats()
    best_loss = float('inf')
    try:
        if method_name == "Dual-Stream":
            X_sales_tr, X_view_tr, _, y_tr, y_log_tr, cat_tr, mask_tr = \
                processor.prepare_features_fair(train_data, label_encoder, is_train=True)
            X_sales_te, X_view_te, _, y_te, y_log_te, cat_te, mask_te = \
                processor.prepare_features_fair(test_data, label_encoder, is_train=False)
            processor.view_scaler = StandardScaler()
            X_sales_tr = torch.FloatTensor(processor.sales_scaler.fit_transform(X_sales_tr)).to(DEVICE)
            X_view_tr = torch.FloatTensor(processor.view_scaler.fit_transform(X_view_tr)).to(DEVICE)
            X_sales_te = torch.FloatTensor(processor.sales_scaler.transform(X_sales_te)).to(DEVICE)
            X_view_te = torch.FloatTensor(processor.view_scaler.transform(X_view_te)).to(DEVICE)
        else:
            _, _, X_all_tr, y_tr, y_log_tr, cat_tr, mask_tr = \
                processor.prepare_features_fair(train_data, label_encoder, is_train=True)
            _, _, X_all_te, y_te, y_log_te, cat_te, mask_te = \
                processor.prepare_features_fair(test_data, label_encoder, is_train=False)
            X_all_tr = torch.FloatTensor(processor.scaler.fit_transform(X_all_tr)).to(DEVICE)
            X_all_te = torch.FloatTensor(processor.scaler.transform(X_all_te)).to(DEVICE)
        y_tr_t = torch.FloatTensor(y_tr).to(DEVICE); y_te_t = torch.FloatTensor(y_te).to(DEVICE)
        cat_tr_t = torch.LongTensor(cat_tr).to(DEVICE); cat_te_t = torch.LongTensor(cat_te).to(DEVICE)
        mask_tr_t = torch.BoolTensor(mask_tr).to(DEVICE); mask_te_t = torch.BoolTensor(mask_te).to(DEVICE)
        model = model.to(DEVICE)
        def weighted_mse_loss(pred, target):
            weights = 1.0 / (target * processor.y_log_max + 1.0); weights = weights / weights.mean()
            return ((pred - target) ** 2 * weights).mean()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        patience_counter = 0; best_model = None; train_start_time = time.time()
        peak_memory_mb = 0.0
        for epoch in range(epochs):
            model.train(); optimizer.zero_grad()
            if method_name == "Dual-Stream": pred = model(X_sales_tr, X_view_tr, cat_tr_t, mask_tr_t)
            elif method_name == "Adaptive": pred = model(X_all_tr, cat_tr_t, mask_tr_t)
            else: pred = model(X_all_tr, cat_tr_t)
            loss = weighted_mse_loss(pred, y_tr_t); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            if epoch % 5 == 0:
                model.eval()
                with torch.no_grad():
                    if method_name == "Dual-Stream": val_pred = model(X_sales_te, X_view_te, cat_te_t, mask_te_t)
                    elif method_name == "Adaptive": val_pred = model(X_all_te, cat_te_t, mask_te_t)
                    else: val_pred = model(X_all_te, cat_te_t)
                    val_loss = weighted_mse_loss(val_pred, y_te_t)
                    if val_loss < best_loss:
                        best_loss = val_loss; patience_counter = 0; best_model = model.state_dict().copy()
                    else: patience_counter += 5
                    if patience_counter >= 15: break
        training_time = time.time() - train_start_time; peak_memory_mb = get_peak_gpu_memory_mb()
        if best_model is not None: model.load_state_dict(best_model)
        model.eval()
        with torch.no_grad():
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inference_start_time = time.time()
            if method_name == "Dual-Stream":
                final_pred = model(X_sales_te, X_view_te, cat_te_t, mask_te_t)
            elif method_name == "Adaptive":
                final_pred = model(X_all_te, cat_te_t, mask_te_t)
            else:
                final_pred = model(X_all_te, cat_te_t)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inference_time_total = time.time() - inference_start_time
            inference_time_ms = (inference_time_total / len(y_te_t)) * 1000

        pred_orig = np.expm1(final_pred.cpu().numpy() * processor.y_log_max)
        true_orig = np.expm1(y_log_te)
        pred_orig = np.maximum(pred_orig, 0.1)
        true_orig = np.maximum(true_orig, 0.1)

        results = get_metrics(final_pred.cpu().numpy(), y_log_te, processor)
        results['val_loss'] = best_loss.item()
        return {
            'metrics': results,
            'train_time_sec': training_time,
            'peak_memory_mb': peak_memory_mb,
            'inference_time_ms_per_sample': inference_time_ms,
            'y_pred_orig': pred_orig,
            'y_true_orig': true_orig
        }
    except Exception as e:
        if verbose: print(f"Training error: {e}")
        return {'metrics': {'r2': -np.inf, 'mae': np.inf, 'mape': np.inf, 'val_loss': np.inf}}

def train_script2_model_run(model, processor, train_data, test_data, method_name, seed,
                           label_encoder, n_categories, sales_dim, input_dim,
                           lr=0.002, weight_decay=0.01,
                           epochs=200, verbose=False):
    set_all_seeds(seed); reset_gpu_stats()
    best_val_loss = float('inf')
    try:
        X_sales_tr, _, X_all_tr, y_tr, y_log_tr, cat_tr, mask_tr = \
            processor.prepare_features_fair(train_data, label_encoder, is_train=True)
        X_sales_te, _, X_all_te, y_te, y_log_te, cat_te, mask_te = \
            processor.prepare_features_fair(test_data, label_encoder, is_train=False)
        X_sales_tr = torch.FloatTensor(processor.sales_scaler.fit_transform(X_sales_tr)).to(DEVICE)
        X_all_tr = torch.FloatTensor(processor.scaler.fit_transform(X_all_tr)).to(DEVICE)
        X_sales_te = torch.FloatTensor(processor.sales_scaler.transform(X_sales_te)).to(DEVICE)
        X_all_te = torch.FloatTensor(processor.scaler.transform(X_all_te)).to(DEVICE)
        y_tr_t = torch.FloatTensor(y_tr).to(DEVICE); y_te_t = torch.FloatTensor(y_te).to(DEVICE)
        cat_tr_t = torch.LongTensor(cat_tr).to(DEVICE); cat_te_t = torch.LongTensor(cat_te).to(DEVICE)
        mask_tr_t = torch.BoolTensor(mask_tr).to(DEVICE); mask_te_t = torch.BoolTensor(mask_te).to(DEVICE)
        model = model.to(DEVICE)
        def weighted_huber_loss(pred, target):
            weights = 1.0 / (target * processor.y_log_max + 1.0); weights = weights / weights.mean()
            huber_per_sample = torch.abs(pred - target); delta = 1.0
            mask_small = huber_per_sample <= delta
            loss_per_sample = torch.where(mask_small, 0.5 * huber_per_sample ** 2, delta * (huber_per_sample - 0.5 * delta))
            return (loss_per_sample * weights).mean()
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999), eps=1e-8)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=0.0001)
        patience_counter = 0; best_model_state = None; train_start_time = time.time()
        for epoch in range(epochs):
            model.train(); optimizer.zero_grad()
            pred = model(X_sales_tr, X_all_tr, cat_tr_t, mask_tr_t)
            loss = weighted_huber_loss(pred, y_tr_t); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5); optimizer.step()
            model.eval()
            with torch.no_grad():
                val_pred = model(X_sales_te, X_all_te, cat_te_t, mask_te_t)
                val_loss = weighted_huber_loss(val_pred, y_te_t)
            scheduler.step(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss; patience_counter = 0; best_model_state = model.state_dict().copy()
            else: patience_counter += 1
            if patience_counter >= 30: break
        training_time = time.time() - train_start_time; peak_memory_mb = get_peak_gpu_memory_mb()
        if best_model_state is not None: model.load_state_dict(best_model_state)
        model.eval()
        with torch.no_grad():
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inference_start_time = time.time()
            final_pred = model(X_sales_te, X_all_te, cat_te_t, mask_te_t)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inference_time_total = time.time() - inference_start_time
            inference_time_ms = (inference_time_total / len(y_te_t)) * 1000

        pred_orig = np.expm1(final_pred.cpu().numpy() * processor.y_log_max)
        true_orig = np.expm1(y_log_te)
        pred_orig = np.maximum(pred_orig, 0.1)
        true_orig = np.maximum(true_orig, 0.1)

        results = get_metrics(final_pred.cpu().numpy(), y_log_te, processor)
        results['val_loss'] = best_val_loss.item()
        return {
            'metrics': results,
            'train_time_sec': training_time,
            'peak_memory_mb': peak_memory_mb,
            'inference_time_ms_per_sample': inference_time_ms,
            'y_pred_orig': pred_orig,
            'y_true_orig': true_orig
        }
    except Exception as e:
        if verbose: print(f"Training error: {e}")
        return {'metrics': {'r2': -np.inf, 'mae': np.inf, 'mape': np.inf, 'val_loss': np.inf}}

def train_xgboost_baseline(train_data, test_data, processor, seed, label_encoder,
                           xgb_params):
    np.random.seed(seed)
    X_sales_tr, _, X_all_tr, _, y_log_tr, cat_tr, _ = \
        processor.prepare_features_fair(train_data, label_encoder, is_train=True)
    X_sales_te, _, X_all_te, _, y_log_te, cat_te, _ = \
        processor.prepare_features_fair(test_data, label_encoder, is_train=False)
    X_train = np.column_stack([X_all_tr, cat_tr]); X_test = np.column_stack([X_all_te, cat_te])
    xgb_scaler = StandardScaler(); X_train_scaled = xgb_scaler.fit_transform(X_train)
    X_test_scaled = xgb_scaler.transform(X_test)

    model = xgb.XGBRegressor(
        **xgb_params,
        random_state=seed,
        n_jobs=-1,
        verbosity=0
    )

    train_start_time = time.time(); model.fit(X_train_scaled, y_log_tr)
    training_time = time.time() - train_start_time
    inference_start_time = time.time(); y_pred_log = model.predict(X_test_scaled)
    inference_time_total = time.time() - inference_start_time
    inference_time_ms = (inference_time_total / len(y_log_te)) * 1000
    pred_orig = np.expm1(y_pred_log); true_orig = np.expm1(y_log_te)
    pred_orig = np.maximum(pred_orig, 0.1); true_orig = np.maximum(true_orig, 0.1)
    results = {'r2': r2_score(true_orig, pred_orig), 'mae': mean_absolute_error(true_orig, pred_orig),
               'mape': np.median(np.abs((true_orig - pred_orig) / np.maximum(true_orig, 1))) * 100}
    return {'metrics': results, 'train_time_sec': training_time, 'peak_memory_mb': 0.0,
            'inference_time_ms_per_sample': inference_time_ms, 'y_pred_log': y_pred_log, 'y_true_log': y_log_te}

def plot_actual_vs_predicted(all_predictions, summary_stats):
    """Creates actual vs predicted plots in Figure 5 style"""

    Path('results').mkdir(exist_ok=True)

    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

    method_order = ['Baseline', 'Dual-Stream', 'Adaptive', 'ConfigurableEnsemble']
    titles = {
        'Baseline': 'Baseline\nR² = {r2:.3f}, MAPE = {mape:.1f}%',
        'Dual-Stream': 'Dual-Stream\nR² = {r2:.3f}, MAPE = {mape:.1f}%',
        'Adaptive': 'Adaptive\nR² = {r2:.3f}, MAPE = {mape:.1f}%',
        'ConfigurableEnsemble': 'Ensemble\nR² = {r2:.3f}, MAPE = {mape:.1f}%'
    }

    positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

    for idx, (method, pos) in enumerate(zip(method_order, positions)):
        if method not in all_predictions or len(all_predictions[method]['y_true']) == 0:
            continue

        ax = fig.add_subplot(gs[pos[0], pos[1]])

        y_true = np.array(all_predictions[method]['y_true'])
        y_pred = np.array(all_predictions[method]['y_pred'])

        scatter = ax.scatter(y_true, y_pred, alpha=0.5, s=30,
                           c=y_true, cmap='viridis', edgecolors='none')

        max_val = max(y_true.max(), y_pred.max())
        min_val = min(y_true.min(), y_pred.min())
        ax.plot([min_val, max_val], [min_val, max_val],
               'r--', linewidth=2, alpha=0.8, label='Perfect Prediction')

        r2 = summary_stats[method]['r2_mean']
        mape = summary_stats[method]['mape_mean']

        ax.set_title(titles[method].format(r2=r2, mape=mape),
                    fontsize=11, fontweight='bold')

        ax.set_xlabel('Actual Sales Quantity', fontsize=10)
        ax.set_ylabel('Predicted Sales Quantity', fontsize=10)

        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

        if pos[1] == 1:
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label('Actual Value', fontsize=9)

        ax.set_aspect('equal', adjustable='box')

    plt.suptitle('Actual vs. Predicted Sales Quantities Comparison',
                fontsize=14, fontweight='bold', y=0.98)

    plt.savefig('results/figure5_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
    print("\nFigure 5 graph saved to 'results/figure5_actual_vs_predicted.png'.")
    plt.close()

def run_faz1_kfold_tournament(K, full_data, label_encoder, model_definitions, model_dims, n_categories, stratify_groups,
                              optimal_params):

    print("\n" + "="*80)
    print(f"PHASE 1: K-FOLD TRANSFORMER TOURNAMENT (K={K}) STARTING")
    print(f"   Using optimized parameters...")
    print("="*80)

    kfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=CV_RANDOM_STATE)
    fold_results_agg = {name: [] for name in model_definitions if name != 'XGBoost'}

    all_predictions = {name: {'y_true': [], 'y_pred': []} for name in fold_results_agg.keys()}

    sales_dim, view_dim, input_dim = model_dims

    for fold, (train_idx, test_idx) in enumerate(kfold.split(full_data, stratify_groups), 1):
        print(f"\n--- Fold {fold}/{K} ---")
        train_data = full_data.iloc[train_idx]
        test_data = full_data.iloc[test_idx]

        for model_name, model_info in fold_results_agg.items():
            print(f"   Training model: {model_name}")

            params = optimal_params[model_name]
            model_kwargs = copy.deepcopy(model_definitions[model_name]['base_kwargs'])

            model_kwargs['d_model'] = params.get('d_model', 32)
            model_kwargs['dropout'] = params.get('dropout', 0.1)

            if model_name == 'Adaptive' and 'alpha' in params:
                model_kwargs['alpha'] = params['alpha']
            if model_name == 'ConfigurableEnsemble' and 'category_emb_dim' in params:
                model_kwargs['category_emb_dim'] = params['category_emb_dim']

            model_instance = model_definitions[model_name]['class'](**model_kwargs)
            processor = ImprovedDataProcessor()

            try:
                if model_name == 'ConfigurableEnsemble':
                    run_metrics = model_definitions[model_name]['train_fn'](
                        model_instance, processor, train_data, test_data, model_name, CV_RANDOM_STATE + fold,
                        label_encoder, n_categories, sales_dim, input_dim,
                        lr=params['lr'], weight_decay=params['weight_decay']
                    )
                else:
                    run_metrics = model_definitions[model_name]['train_fn'](
                        model_instance, processor, train_data, test_data, model_name, CV_RANDOM_STATE + fold,
                        label_encoder,
                        lr=params['lr'], weight_decay=params['weight_decay']
                    )

                fold_results_agg[model_name].append(run_metrics)

                if 'y_pred_orig' in run_metrics and 'y_true_orig' in run_metrics:
                    all_predictions[model_name]['y_true'].extend(run_metrics['y_true_orig'])
                    all_predictions[model_name]['y_pred'].extend(run_metrics['y_pred_orig'])

                print(f"       Result: R2={run_metrics['metrics']['r2']:.4f} (Time: {run_metrics['train_time_sec']:.1f}s)")
            except Exception as e:
                print(f"       ERROR: {model_name} (Fold: {fold}) could not be trained. Error: {e}")
                fold_results_agg[model_name].append(None)

    print("\n" + "="*80)
    print(f"PHASE 1 (K={K}) SUMMARY STATISTICS")
    print("="*80)

    summary_stats = {}
    for model_name, fold_runs in fold_results_agg.items():
        valid_runs = [r for r in fold_runs if r is not None and r['metrics']['r2'] > -np.inf]
        if not valid_runs:
            print(f"\nModel: {model_name} (NO SUCCESSFUL FOLDS)")
            continue

        summary = {'method': model_name}
        for key in valid_runs[0]['metrics'].keys():
            if key == 'val_loss': continue
            summary[f'{key}_values'] = [r['metrics'][key] for r in valid_runs]
        summary['train_values'] = [r['train_time_sec'] for r in valid_runs]
        summary['peak_values'] = [r['peak_memory_mb'] for r in valid_runs]
        summary['inference_values'] = [r['inference_time_ms_per_sample'] * 1000 for r in valid_runs]

        for key in list(summary.keys()):
            if key.endswith('_values'):
                metric_name = key.replace('_values', '')
                values = summary[key]
                summary[f'{metric_name}_mean'] = np.mean(values)
                summary[f'{metric_name}_std'] = np.std(values, ddof=1)

        summary_stats[model_name] = summary
        print(f"\nModel: {model_name} ({len(valid_runs)}/{K} successful folds)")
        print(f"   R2 (Mean): {summary['r2_mean']:.4f} +/- {summary['r2_std']:.4f}")
        print(f"   MAE (Mean): {summary['mae_mean']:.2f} +/- {summary['mae_std']:.2f}")
        print(f"   MAPE (Mean): {summary['mape_mean']:.1f} +/- {summary['mape_std']:.1f}%")
        print(f"   Training Time (Mean): {summary['train_mean']:.2f} +/- {summary['train_std']:.2f} s")
        print(f"   Peak Memory (Mean): {summary['peak_mean']:.2f} +/- {summary['peak_std']:.2f} MB")
        print(f"   Inference Time (Mean): {summary['inference_mean']:.4f} +/- {summary['inference_std']:.4f} ms/sample")

    best_transformer_method, significance_markers = compute_significance_kfold(summary_stats)
    generate_extended_publication_table(summary_stats, significance_markers, K)

    plot_actual_vs_predicted(all_predictions, summary_stats)

    return best_transformer_method, summary_stats

def run_faz2_kfold_final(K, champion_model_name, champion_model_info,
                         full_data, label_encoder, model_dims, n_categories, stratify_groups,
                         optimal_params):

    print("\n" + "="*80)
    print(f"PHASE 2: K-FOLD CHAMPION vs. XGBOOST (K={K})")
    print(f"   Champion Model: {champion_model_name}")
    print(f"   Using optimized parameters...")
    print("="*80)

    sales_dim, view_dim, input_dim = model_dims
    kfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=CV_RANDOM_STATE)

    general_results_champion = []
    general_results_xgb = []
    specialized_results = []

    champion_train_fn = champion_model_info['train_fn']
    params_champ = optimal_params[champion_model_name]
    params_xgb = optimal_params['XGBoost']

    model_kwargs_champ = copy.deepcopy(champion_model_info['base_kwargs'])
    model_kwargs_champ['d_model'] = params_champ.get('d_model', 32)
    model_kwargs_champ['dropout'] = params_champ.get('dropout', 0.1)
    if 'category_emb_dim' in params_champ:
        model_kwargs_champ['category_emb_dim'] = params_champ['category_emb_dim']
    if champion_model_name == 'Adaptive' and 'alpha' in params_champ:
        model_kwargs_champ['alpha'] = params_champ['alpha']

    for fold, (train_idx, test_idx) in enumerate(kfold.split(full_data, stratify_groups), 1):
        print(f"\n--- Fold {fold}/{K} ---")
        train_data = full_data.iloc[train_idx]
        test_data = full_data.iloc[test_idx]

        print(f"   Testing: GENERAL (n={len(test_data)})")
        proc_gen_champ = ImprovedDataProcessor(); proc_gen_xgb = ImprovedDataProcessor()

        model_instance_gen = champion_model_info['class'](**model_kwargs_champ)
        if champion_model_name == 'ConfigurableEnsemble':
            champ_gen_metrics = champion_train_fn(
                model_instance_gen, proc_gen_champ, train_data, test_data, champion_model_name, CV_RANDOM_STATE + fold,
                label_encoder, n_categories, sales_dim, input_dim,
                lr=params_champ['lr'], weight_decay=params_champ['weight_decay']
            )
        else:
            champ_gen_metrics = champion_train_fn(
                model_instance_gen, proc_gen_champ, train_data, test_data, champion_model_name, CV_RANDOM_STATE + fold,
                label_encoder,
                lr=params_champ['lr'], weight_decay=params_champ['weight_decay']
            )
        general_results_champion.append(champ_gen_metrics)

        xgb_gen_metrics = train_xgboost_baseline(
            train_data, test_data, proc_gen_xgb, CV_RANDOM_STATE + fold, label_encoder,
            xgb_params=params_xgb
        )
        general_results_xgb.append(xgb_gen_metrics)

        print(f"      {champion_model_name}: R2={champ_gen_metrics['metrics']['r2']:.4f}, XGBoost: R2={xgb_gen_metrics['metrics']['r2']:.4f}")

        has_view_data = test_data['view_count'].fillna(0) > 0
        complete_data = test_data[has_view_data].copy(); missing_data = test_data[~has_view_data].copy()
        if len(complete_data) == 0 or len(missing_data) == 0:
            print(f"   WARNING: SPECIALIZED ANALYSIS SKIPPED (Fold {fold}) - Insufficient data separation.")
            specialized_results.append(None); continue

        proc_comp_champ = ImprovedDataProcessor(); proc_comp_xgb = ImprovedDataProcessor()
        model_inst_comp = champion_model_info['class'](**model_kwargs_champ)
        if champion_model_name == 'ConfigurableEnsemble':
            champ_metrics_c = champion_train_fn(model_inst_comp, proc_comp_champ, train_data, complete_data, champion_model_name, CV_RANDOM_STATE + fold, label_encoder, n_categories, sales_dim, input_dim, lr=params_champ['lr'], weight_decay=params_champ['weight_decay'])
        else:
            champ_metrics_c = champion_train_fn(model_inst_comp, proc_comp_champ, train_data, complete_data, champion_model_name, CV_RANDOM_STATE + fold, label_encoder, lr=params_champ['lr'], weight_decay=params_champ['weight_decay'])
        xgb_metrics_c = train_xgboost_baseline(train_data, complete_data, proc_comp_xgb, CV_RANDOM_STATE + fold, label_encoder, xgb_params=params_xgb)

        proc_miss_champ = ImprovedDataProcessor(); proc_miss_xgb = ImprovedDataProcessor()
        model_inst_miss = champion_model_info['class'](**model_kwargs_champ)
        if champion_model_name == 'ConfigurableEnsemble':
            champ_metrics_m = champion_train_fn(model_inst_miss, proc_miss_champ, train_data, missing_data, champion_model_name, CV_RANDOM_STATE + fold, label_encoder, n_categories, sales_dim, input_dim, lr=params_champ['lr'], weight_decay=params_champ['weight_decay'])
        else:
            champ_metrics_m = champion_train_fn(model_inst_miss, proc_miss_champ, train_data, missing_data, champion_model_name, CV_RANDOM_STATE + fold, label_encoder, lr=params_champ['lr'], weight_decay=params_champ['weight_decay'])
        xgb_metrics_m = train_xgboost_baseline(train_data, missing_data, proc_miss_xgb, CV_RANDOM_STATE + fold, label_encoder, xgb_params=params_xgb)

        gap_complete = champ_metrics_c['metrics']['r2'] - xgb_metrics_c['metrics']['r2']
        gap_missing = champ_metrics_m['metrics']['r2'] - xgb_metrics_m['metrics']['r2']

        print(f"   Testing: SPECIALIZED (Complete Data): {champion_model_name} R2={champ_metrics_c['metrics']['r2']:.4f}, XGB R2={xgb_metrics_c['metrics']['r2']:.4f}, Diff={gap_complete:+.4f}")
        print(f"   Testing: SPECIALIZED (Missing Data): {champion_model_name} R2={champ_metrics_m['metrics']['r2']:.4f}, XGB R2={xgb_metrics_m['metrics']['r2']:.4f}, Diff={gap_missing:+.4f}")

        specialized_results.append({
            'complete_champ_r2': champ_metrics_c['metrics']['r2'], 'complete_xgb_r2': xgb_metrics_c['metrics']['r2'], 'complete_gap': gap_complete,
            'missing_champ_r2': champ_metrics_m['metrics']['r2'], 'missing_xgb_r2': xgb_metrics_m['metrics']['r2'], 'missing_gap': gap_missing,
        })

    analyze_faz2_results(champion_model_name, general_results_champion, general_results_xgb, specialized_results, K)

def compute_significance_kfold(results_dict):
    print(f"\n{'='*60}")
    print("PHASE 1: STATISTICAL SIGNIFICANCE TEST (Based on R2)")
    print(f"{'='*60}")
    methods = list(results_dict.keys())
    if not methods: return None, {}
    best_r2_method = max(methods, key=lambda m: results_dict[m]['r2_mean'])
    print(f"\nBest Model (By R2 Mean): {best_r2_method}")
    print(f"   R2 = {results_dict[best_r2_method]['r2_mean']:.4f} +/- {results_dict[best_r2_method]['r2_std']:.4f}")
    print(f"\nPaired t-tests (vs {best_r2_method}):")
    print(f"{'Method':<25} {'Mean R2':<12} {'p-value':<12} {'Significance':<15}")
    print("-" * 64)
    significance_markers = {}
    best_scores = results_dict[best_r2_method]['r2_values']
    for method in methods:
        if method == best_r2_method:
            print(f"{method:<25} {results_dict[method]['r2_mean']:.4f}       -            (reference)")
            significance_markers[method] = ""
        else:
            method_scores = results_dict[method]['r2_values']
            if len(best_scores) != len(method_scores):
                print(f"{method:<25} (Score count mismatch, test skipped)")
                continue
            t_stat, p_value = stats.ttest_rel(best_scores, method_scores)

            if p_value < 0.001: marker = "***"; sig_text = "p < 0.001"
            elif p_value < 0.01: marker = "**"; sig_text = "p < 0.01"
            elif p_value < 0.05: marker = "*"; sig_text = "p < 0.05"
            else: marker = "ns"; sig_text = "not significant"

            if np.mean(best_scores) < np.mean(method_scores) and p_value < 0.05:
                sig_text = "ANOMALY: Lower R2!"
                marker = "???"

            significance_markers[method] = marker if marker != "ns" else ""
            print(f"{method:<25} {results_dict[method]['r2_mean']:.4f}       {p_value:.4f}       {sig_text}")
    return best_r2_method, significance_markers

def generate_extended_publication_table(results_dict, significance_markers, K):
    print(f"\n{'='*80}")
    print(f"PHASE 1 (K={K}): PUBLICATION TABLE (Optimized)")
    print(f"{'='*80}\n")
    print(f"{'Method':<25} {'R2 (mean+/-std)':<20} {'MAPE (%)':<18} {'MAE':<18} {'Training (s)':<15} {'Memory (MB)':<15} {'Inference (ms)':<15}")
    print("-" * 130)
    method_order = ['Baseline', 'Dual-Stream', 'Adaptive', 'ConfigurableEnsemble']
    for method in method_order:
        if method in results_dict:
            r = results_dict[method]
            sig = significance_markers.get(method, '')
            print(f"{method:<25} "
                  f"{r['r2_mean']:.4f} +/- {r['r2_std']:.4f}{sig:<3}  "
                  f"{r['mape_mean']:.1f} +/- {r['mape_std']:.1f}      "
                  f"{r['mae_mean']:.2f} +/- {r['mae_std']:.2f}   "
                  f"{r['train_mean']:.2f} +/- {r['train_std']:.2f}   "
                  f"{r['peak_mean']:.2f} +/- {r['peak_std']:.2f}   "
                  f"{r['inference_mean']:.4f} +/- {r['inference_std']:.4f}")
    print(f"\nSignificance (* p<0.05, ** p<0.01, *** p<0.001) vs. best R2 model (Paired t-test, K={K})")

def analyze_faz2_results(champion_model_name, general_results_champion, general_results_xgb, specialized_results, K):
    print("\n" + "="*80)
    print(f"PHASE 2A (K={K}): GENERAL PERFORMANCE SUMMARY ({champion_model_name} vs. XGBoost)")
    print("="*80)
    summary_list = []
    for name, results_list in [(champion_model_name, general_results_champion), ("XGBoost", general_results_xgb)]:
        valid_runs = [r for r in results_list if r is not None and r['metrics']['r2'] > -np.inf]
        if not valid_runs: continue
        summary = {'method': name}
        for key in valid_runs[0]['metrics'].keys():
             if key == 'val_loss': continue
             summary[f'{key}_values'] = [r['metrics'][key] for r in valid_runs]
        summary['train_values'] = [r['train_time_sec'] for r in valid_runs]
        for key in list(summary.keys()):
            if key.endswith('_values'):
                metric_name = key.replace('_values', '')
                values = summary[key]
                summary[f'{metric_name}_mean'] = np.mean(values)
                summary[f'{metric_name}_std'] = np.std(values, ddof=1)
        summary_list.append(summary)
    print(f"{'Method':<25} {'R2 (mean+/-std)':<20} {'MAPE (%)':<18} {'MAE':<18} {'Training (s)':<15}")
    print("-" * 97)
    for r in summary_list:
        print(f"{r['method']:<25} "
              f"{r['r2_mean']:.4f} +/- {r['r2_std']:.4f}   "
              f"{r['mape_mean']:.1f} +/- {r['mape_std']:.1f}      "
              f"{r['mae_mean']:.2f} +/- {r['mae_std']:.2f}   "
              f"{r['train_mean']:.2f} +/- {r['train_std']:.2f}")
    df_spec = pd.DataFrame([r for r in specialized_results if r is not None])
    if df_spec.empty:
        print("\nPHASE 2B SPECIALIZED ANALYSIS SKIPPED: Insufficient data could be separated.")
        return

    print("\n" + "="*80)
    print(f"PHASE 2B (K={K}): SPECIALIZED PERFORMANCE ANALYSIS (Missing vs. Complete Data R2 Difference)")
    print("="*80)
    print(f"\n--- Raw R2 Scores Summary (K={K} Fold Average) ---")
    print(f"{'Data Type':<15} {'Model':<25} {'Mean R2':<15} {'Std. Dev':<15}")
    print("-" * 72)
    print(f"{'COMPLETE Data':<15} {champion_model_name:<25} {df_spec['complete_champ_r2'].mean():<15.4f} {df_spec['complete_champ_r2'].std():<15.4f}")
    print(f"{'COMPLETE Data':<15} {'XGBoost':<25} {df_spec['complete_xgb_r2'].mean():<15.4f} {df_spec['complete_xgb_r2'].std():<15.4f}")
    print(f"{'MISSING Data':<15} {champion_model_name:<25} {df_spec['missing_champ_r2'].mean():<15.4f} {df_spec['missing_champ_r2'].std():<15.4f}")
    print(f"{'MISSING Data':<15} {'XGBoost':<25} {df_spec['missing_xgb_r2'].mean():<15.4f} {df_spec['missing_xgb_r2'].std():<15.4f}")
    print("\n--- Difference Analysis (Champion - XGBoost) ---")
    complete_avg_gap = df_spec['complete_gap'].mean()
    complete_std_gap = df_spec['complete_gap'].std()
    missing_avg_gap = df_spec['missing_gap'].mean()
    missing_std_gap = df_spec['missing_gap'].std()
    print(f"{'Data Type':<15} {'Mean Difference (R2)':<25} {'Standard Deviation':<20}")
    print("-" * 60)
    print(f"{'COMPLETE Data':<15} {complete_avg_gap:+.4f}               {complete_std_gap:.4f}")
    print(f"{'MISSING Data':<15} {missing_avg_gap:+.4f}               {missing_std_gap:.4f}")
    complete_wins = (df_spec['complete_gap'] > 0).sum()
    missing_wins = (df_spec['missing_gap'] > 0).sum()
    print(f"\n{'Win Rate':<15} ({champion_model_name} > XGBoost)")
    print(f"   COMPLETE Data: {complete_wins}/{len(df_spec)} ({complete_wins/len(df_spec)*100:.0f}%)")
    print(f"   MISSING Data: {missing_wins}/{len(df_spec)} ({missing_wins/len(df_spec)*100:.0f}%)")
    print("\n--- Statistical Significance (Difference vs 0) ---")
    t_complete, p_complete = stats.ttest_1samp(df_spec['complete_gap'], 0)
    t_missing, p_missing = stats.ttest_1samp(df_spec['missing_gap'], 0)

    def get_sig_marker(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return "ns"

    print(f"   COMPLETE Data: p-value = {p_complete:.4f} ({get_sig_marker(p_complete)})")
    print(f"   MISSING Data: p-value = {p_missing:.4f} ({get_sig_marker(p_missing)})")

    t_diff, p_diff = stats.ttest_rel(df_spec['complete_gap'], df_spec['missing_gap'])
    print(f"\n--- COMPLETE vs MISSING Difference Comparison ---")
    print(f"   Is the difference between the two differences significant? p-value = {p_diff:.4f} ({get_sig_marker(p_diff)})")
    print("\n" + "="*80)
    print(f"PHASE 2B (K={K}): SPECIALIZED ANALYSIS DECISION")
    print("="*80)
    if p_complete < 0.05 and complete_avg_gap > 0: print(f"1. COMPLETE DATA: {champion_model_name} SIGNIFICANTLY BETTER (p={p_complete:.4f})")
    elif p_complete < 0.05 and complete_avg_gap < 0: print(f"1. COMPLETE DATA: XGBoost SIGNIFICANTLY BETTER (p={p_complete:.4f})")
    else: print(f"1. COMPLETE DATA: NO SIGNIFICANT DIFFERENCE (p={p_complete:.4f})")
    if p_missing < 0.05 and missing_avg_gap > 0: print(f"\n2. MISSING DATA: {champion_model_name} SIGNIFICANTLY BETTER (p={p_missing:.4f})")
    elif p_missing < 0.05 and missing_avg_gap < 0: print(f"\n2. MISSING DATA: XGBoost SIGNIFICANTLY BETTER (p={p_missing:.4f})")
    else: print(f"\n2. MISSING DATA: NO SIGNIFICANT DIFFERENCE (p={p_missing:.4f})")
    if p_diff < 0.05:
        if complete_avg_gap > missing_avg_gap: print(f"\n3. VIEW DATA ADVANTAGE: {champion_model_name} benefits SIGNIFICANTLY MORE from view data.")
        else: print(f"\n3. VIEW DATA ADVANTAGE: {champion_model_name}'s advantage is SIGNIFICANTLY GREATER when data is MISSING.")
    else: print(f"\n3. VIEW DATA ADVANTAGE: Advantage is NOT SIGNIFICANTLY affected by whether data is missing or complete.")

def main(excel_path='restaurant_merged_data.xlsx'):

    print("="*80)
    print("STAGE 2: K-FOLD EVALUATION (WITH OPTIMIZED PARAMETERS)")
    print(f"Device in use: {DEVICE}")
    print("="*80)

    optimal_params_from_hpo = {
        "Baseline": {
            "lr": 0.01,
            "d_model": 64,
            "dropout": 0.1,
            "weight_decay": 1e-05
        },
        "Dual-Stream": {
            "lr": 0.01,
            "d_model": 64,
            "dropout": 0.3,
            "weight_decay": 1e-05
        },
        "Adaptive": {
            "lr": 0.01,
            "d_model": 64,
            "dropout": 0.1,
            "weight_decay": 1e-05,
            "alpha": 0.5
        },
        "ConfigurableEnsemble": {
            "lr": 0.005,
            "d_model": 64,
            "category_emb_dim": 16,
            "dropout": 0.1,
            "weight_decay": 0.0001
        },
        "XGBoost": {
            "subsample": 0.8,
            "n_estimators": 100,
            "max_depth": 5,
            "learning_rate": 0.1,
            "colsample_bytree": 0.8
        }
    }

    print("The following optimized parameters will be used for K-Fold testing:")
    print(json.dumps(optimal_params_from_hpo, indent=2))

    print("\nLoading and preprocessing data...")
    data_loader = ImprovedDataProcessor()
    full_data = data_loader.load_full_dataset(excel_path)
    all_categories = full_data['sales_main_category'].fillna('Unknown').astype(str).str.strip()
    label_encoder = LabelEncoder()
    label_encoder.fit(all_categories)
    n_categories = len(label_encoder.classes_)

    stratify_groups = data_loader.get_stratify_groups(full_data, test_size_for_rare_calc=1/K_FOLDS)

    temp_processor = ImprovedDataProcessor()
    X_sales_temp, X_view_temp, X_all_temp, _, _, _, _ = temp_processor.prepare_features_fair(
        full_data.head(), label_encoder, is_train=True)
    model_dims = (X_sales_temp.shape[1], X_view_temp.shape[1], X_all_temp.shape[1])
    sales_dim, view_dim, input_dim = model_dims

    print(f"Number of categories: {n_categories}")
    print(f"Feature dimensions (Sales: {sales_dim}, View: {view_dim}, All: {input_dim})")

    model_definitions = {
        'Baseline': {
            'class': BaselineTransformer,
            'base_kwargs': {'input_dim': input_dim, 'n_categories': n_categories, 'nhead': 4},
            'train_fn': train_script1_model_run
        },
        'Dual-Stream': {
            'class': DualStreamTransformer,
            'base_kwargs': {'sales_dim': sales_dim, 'view_dim': view_dim, 'n_categories': n_categories, 'nhead': 4},
            'train_fn': train_script1_model_run
        },
        'Adaptive': {
            'class': AdaptiveWeightingTransformer,
            'base_kwargs': {'input_dim': input_dim, 'n_categories': n_categories, 'nhead': 4},
            'train_fn': train_script1_model_run
        },
        'ConfigurableEnsemble': {
            'class': ConfigurableEnsembleTransformer,
            'base_kwargs': {'sales_dim': sales_dim, 'full_dim': input_dim, 'n_categories': n_categories, 'nhead': 4, 'num_layers': 1},
            'train_fn': train_script2_model_run,
        }
    }

    best_transformer_5, faz1_results_5 = run_faz1_kfold_tournament(
        K_FOLDS, full_data, label_encoder, model_definitions, model_dims, n_categories, stratify_groups,
        optimal_params_from_hpo
    )
    if best_transformer_5:
        run_faz2_kfold_final(
            K_FOLDS, best_transformer_5, model_definitions[best_transformer_5],
            full_data, label_encoder, model_dims, n_categories, stratify_groups,
            optimal_params_from_hpo
        )

    print("\n" + "="*80)
    print(f"K={K_FOLDS} Fold Evaluation Completed!")
    print("="*80)

    Path('results').mkdir(exist_ok=True)
    faz1_save_dict = {method: {k: v for k, v in r.items() if k != 'method'} for method, r in faz1_results_5.items()}
    with open(f'results/faz1_k{K_FOLDS}_fold_OPTIMIZED.json', 'w') as f:
        json.dump(faz1_save_dict, f, indent=2)
    print(f"Phase 1 (K={K_FOLDS}) (Optimized) results saved to 'results/faz1_k{K_FOLDS}_fold_OPTIMIZED.json'.")

if __name__ == "__main__":
    main(excel_path='restaurant_merged_data.xlsx')

Device in use: cuda
STAGE 2: K-FOLD EVALUATION (WITH OPTIMIZED PARAMETERS)
Device in use: cuda
The following optimized parameters will be used for K-Fold testing:
{
  "Baseline": {
    "lr": 0.01,
    "d_model": 64,
    "dropout": 0.1,
    "weight_decay": 1e-05
  },
  "Dual-Stream": {
    "lr": 0.01,
    "d_model": 64,
    "dropout": 0.3,
    "weight_decay": 1e-05
  },
  "Adaptive": {
    "lr": 0.01,
    "d_model": 64,
    "dropout": 0.1,
    "weight_decay": 1e-05,
    "alpha": 0.5
  },
  "ConfigurableEnsemble": {
    "lr": 0.005,
    "d_model": 64,
    "category_emb_dim": 16,
    "dropout": 0.1,
    "weight_decay": 0.0001
  },
  "XGBoost": {
    "subsample": 0.8,
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "colsample_bytree": 0.8
  }
}

Loading and preprocessing data...
Number of categories: 18
Feature dimensions (Sales: 5, View: 5, All: 10)

PHASE 1: K-FOLD TRANSFORMER TOURNAMENT (K=5) STARTING
   Using optimized parameters...

--- Fold 1/5 ---
   Train